# Evans Index Y (EI-y) — MRI Adaptation

Measures the anterior-posterior length of the frontal horns of the lateral ventricles, normalized by the maximum AP diameter of the brain, on axial slices above the Anterior Commissure (AC).

**MRI adaptation**: No HU windowing — Otsu and adaptive thresholding operate directly on T1w intensities.

**Reference**: [Samadani et al., JNS 2024](https://thejns.org/view/journals/j-neurosurg/141/3/article-p822.xml)

## Library imports

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import cv2 as cv
import SimpleITK as sitk
from skimage import measure, filters
import matplotlib.pyplot as plt

from mri_utils import (rotate_image, translate_image, getLargestCC,
                       normalize_to_uint8,
                       OASIS_AC_VOXEL, OASIS_PC_VOXEL, parse_acpc_coords, load_acpc_from_txt)

## Configuration

Set `DATASET` to match your data source. The notebook adapts file discovery and AC/PC coordinate loading accordingly.

Supported datasets:
- **`"oasis1"`** — OASIS-1 cross-sectional (skull-stripped, atlas-registered `.img` files); AC/PC from hardcoded OASIS atlas voxel coordinates
- **`"abcd"`** — ABCD Study T1w scans (ART-corrected `.nii`/`.nii.gz` files); AC/PC from per-subject `.txt` files
- **`"custom"`** — Any NIfTI dataset; set `SCAN_GLOB` to your file pattern

In [ ]:
# ── Choose your dataset ──────────────────────────────────────────────
DATASET = "oasis1"  # "oasis1", "abcd", or "custom"

# ── Dataset-specific settings ────────────────────────────────────────
if DATASET == "oasis1":
    DATA_DIR = "data/oasis1/disc1"
    DEMOGRAPHICS_PATH = "data/oasis1/oasis_cross-sectional-5708aa0a98d82080.xlsx"
    # OASIS skull-stripped, gain-field corrected, atlas-registered volumes
    SCAN_GLOB = "OAS1_*/PROCESSED/MPRAGE/T88_111/*_masked_gfc.img"
    SUBJECT_ID_FROM_PATH = lambda p: p.split(os.sep)[-5]  # e.g. OAS1_0001_MR1

elif DATASET == "abcd":
    DATA_DIR = "data/abcd"
    DEMOGRAPHICS_PATH = None  # set path to ABCD demographics CSV/TSV if available
    # ART-corrected T1w scans (output from ART batch script)
    SCAN_GLOB = "**/*_T1w_acpc.nii*"
    SUBJECT_ID_FROM_PATH = lambda p: os.path.basename(p).split("_ses")[0]  # e.g. sub-NDARINVXXXXXXXX

elif DATASET == "custom":
    DATA_DIR = "data/my_dataset"
    DEMOGRAPHICS_PATH = None
    SCAN_GLOB = "**/*.nii.gz"  # adjust to your file naming
    SUBJECT_ID_FROM_PATH = lambda p: os.path.basename(p).replace(".nii.gz", "").replace(".nii", "")

else:
    raise ValueError(f"Unknown DATASET: {DATASET!r}. Use 'oasis1', 'abcd', or 'custom'.")

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Dataset: {DATASET}")
print(f"Data dir: {DATA_DIR}")
print(f"Scan pattern: {SCAN_GLOB}")

## Processing parameters

In [ ]:
ei_y_processing_params = {
    "c_thresh": 7,
    "dec_thresh": 0,
    "ei_y_num_slices_above_ac_to_eval": 20,
    "axial_gaussian_blur_initial": 3,
    "axial_adaptive_thresh_size": 85,
    "axial_adaptive_thresh_mean_thresh": 7.5,
    "otsu_adaptive_diff_thresh": 0.35,
    "otsu_adaptive_horizontal_thresh": 7,
    "otsu_adaptive_vert_thresh": 40,
    "lv_h_roi_col_thresh_prcnt": 0.15,
    "lv_h_roi_row_thresh": 20,
    "midline_detected_vert_thresh": 1.5,
}

## Core function: calc_ei_y()

In [ ]:
def calc_ei_y(axial_data, ac_i, ac_j, ac_k, ei_y_processing_parameters, visualize=False):
    """
    Calculates the modified y-axis Evans Index (EI_y) from axial brain MRI data.

    This function iterates through axial slices superior to the anterior commissure (AC)
    to evaluate the anterior-posterior (y-axis) length of the frontal horns. It dynamically
    adjusts a column threshold to separate the left and right ventricles. The final EI_y
    is computed as the maximum frontal horn y-length divided by the maximum y-length
    of the brain mask on the optimal slice.

    MRI adaptation: No HU windowing is applied. axial_data is passed in as raw T1w
    intensity values. Otsu and adaptive thresholding operate directly on these intensities.

    Parameters
    ----------
    axial_data : numpy.ndarray
        A 3D array representing the axial MRI volume (raw T1w intensities).
    ac_i : float or int
        The x-coordinate (column index) of the anterior commissure.
    ac_j : float or int
        The y-coordinate (row index) of the anterior commissure.
    ac_k : float or int
        The z-coordinate (slice index) of the anterior commissure.
    ei_y_processing_parameters : dict
        Dictionary of predefined parameters for image processing and ROI definitions.
    visualize : bool, optional
        If True, displays intermediate masks and the final calculated lengths overlaid
        on the optimal slice. Default is False.

    Returns
    -------
    ei_y : float or None
        The calculated y-axis Evans Index. Returns None if a valid measurement
        cannot be found.
    opt_plane : int or None
        The axial slice index (z-coordinate) where the maximum y-length was found.
        Returns None if extraction fails.
    """

    c_thresh = ei_y_processing_parameters["c_thresh"]
    dec_thresh = ei_y_processing_parameters["dec_thresh"]
    ei_y_num_slices_above_ac_to_eval = ei_y_processing_parameters["ei_y_num_slices_above_ac_to_eval"]
    axial_gaussian_blur_initial = ei_y_processing_parameters["axial_gaussian_blur_initial"]
    axial_adaptive_thresh_size = ei_y_processing_parameters["axial_adaptive_thresh_size"]
    axial_adaptive_thresh_mean_thresh = ei_y_processing_parameters["axial_adaptive_thresh_mean_thresh"]
    otsu_adaptive_diff_thresh = ei_y_processing_parameters["otsu_adaptive_diff_thresh"]
    otsu_adaptive_horizontal_thresh = ei_y_processing_parameters["otsu_adaptive_horizontal_thresh"]
    otsu_adaptive_vert_thresh = ei_y_processing_parameters["otsu_adaptive_vert_thresh"]
    lv_h_roi_col_thresh_prcnt = ei_y_processing_parameters["lv_h_roi_col_thresh_prcnt"]
    lv_h_roi_row_thresh = ei_y_processing_parameters["lv_h_roi_row_thresh"]
    midline_detected_vert_thresh = ei_y_processing_parameters["midline_detected_vert_thresh"]

    y2 = int(round(ac_j))
    ac_i_int = int(round(ac_i))

    final_x1_opt = None
    final_y1_opt = None
    full_brain_mask_opt = None
    opt_plane = None
    axial_pl_opt = None

    max_dec_thresh_limit = 20

    while True:
        # Prevent infinite loops if valid ventricle masks are never found
        if dec_thresh > max_dec_thresh_limit:
            print("Warning: Exceeded threshold adjustment limits. Could not calculate EI_y.")
            return None, None

        max_fh_length = 0  # resetting value for each image, each thresh

        for plane in range(int(round(ac_k)), int(round(ac_k)) + ei_y_num_slices_above_ac_to_eval):
            if plane >= axial_data.shape[0]:
                break

            axial_brain_im = axial_data[plane, :, :]

            # Denoising — no windowing; operate directly on raw MRI intensities
            axial_gauss_blurred = cv.GaussianBlur(
                normalize_to_uint8(axial_brain_im),
                (axial_gaussian_blur_initial, axial_gaussian_blur_initial),
                cv.BORDER_DEFAULT
            )
            thresh = filters.threshold_otsu(axial_gauss_blurred)
            axial_bin = axial_gauss_blurred > thresh

            axial_ad_th = cv.adaptiveThreshold(
                axial_gauss_blurred, 1,
                cv.ADAPTIVE_THRESH_MEAN_C,
                cv.THRESH_BINARY,
                axial_adaptive_thresh_size,
                axial_adaptive_thresh_mean_thresh
            )

            # Background subtraction: adaptive threshold classifies background as
            # foreground due to local thresholding — subtract it out
            blobs_labels = measure.label(axial_brain_im, background=0)
            background_pixels = blobs_labels == np.argmax(np.bincount(blobs_labels.flatten()))

            axial_brain_mask = np.where(background_pixels > 0, 0, axial_ad_th)

            # Floodfill from (0,0) to extract the entire brain mask
            im_th = axial_brain_mask.copy()
            h, w = im_th.shape[:2]
            mask = np.zeros((h + 2, w + 2), np.uint8)
            im_floodfill = im_th.copy()
            res = cv.floodFill(np.uint8(im_floodfill), mask, (0, 0), 255)
            full_brain_mask = (1 - res[2]).copy()
            full_brain_mask = full_brain_mask[1:h + 1, 1:w + 1]
            full_brain_mask = np.where(
                getLargestCC(measure.label(full_brain_mask, background=0)) > 0, 1, 0
            )
            [fbr, fbc] = full_brain_mask.nonzero()

            M = cv.moments(np.uint8(full_brain_mask))
            if M["m00"] == 0:
                continue

            cX = int(M["m10"] / M["m00"])
            cY = int(M["m01"] / M["m00"])

            # Primary segmentation via adaptive threshold
            adaptive_ventricles = np.where(full_brain_mask - axial_brain_mask > 0, 1, 0)

            if np.sum(adaptive_ventricles) == 0:
                continue

            tm = np.zeros(adaptive_ventricles.shape)
            tm[:adaptive_ventricles.shape[0] // 2, :] = 1

            ventricles = adaptive_ventricles

            # Combine Otsu and adaptive ventricles if Otsu captures additional
            # tissue meeting the diff criteria
            try:
                otsu_ventricles = getLargestCC(
                    measure.label(np.where(full_brain_mask - axial_bin > 0, 1, 0), background=0)
                )
                if (np.sum(otsu_ventricles * tm) > 0) and (np.sum((adaptive_ventricles) * tm) > 0):
                    diff_otsu_adaptive = np.where(otsu_ventricles - adaptive_ventricles > 0, 1, 0) * tm
                    if np.sum(diff_otsu_adaptive) > 0:
                        Ma = cv.moments(np.uint8(adaptive_ventricles) * tm)
                        aX = int(Ma["m10"] / Ma["m00"])
                        aY = int(Ma["m01"] / Ma["m00"])
                        diff_density = np.sum(diff_otsu_adaptive) / np.sum(otsu_ventricles * tm)
                        Md = cv.moments(np.uint8(diff_otsu_adaptive))
                        dX = int(Md["m10"] / Md["m00"])
                        dY = int(Md["m01"] / Md["m00"])

                        diff_check = (
                            (diff_density > otsu_adaptive_diff_thresh)
                            and (np.abs(aX - dX) < otsu_adaptive_horizontal_thresh)
                            and (np.abs(aY - dY) < otsu_adaptive_vert_thresh)
                        )

                        if diff_check:
                            ventricles = np.logical_or(adaptive_ventricles, diff_otsu_adaptive)
            except (ValueError, IndexError, ZeroDivisionError):
                ventricles = adaptive_ventricles

            # Horizontal ROI to keep left and right FH connected
            h_roi = np.zeros(ventricles.shape)
            [r_v, c_v] = ventricles.nonzero()
            mr_v = (max(r_v) + min(r_v)) // 2
            [fr, fc] = full_brain_mask.nonzero()
            fbc_range = max(fc) - min(fc)
            col_thresh = int(fbc_range * lv_h_roi_col_thresh_prcnt)

            r_start = max(0, mr_v - lv_h_roi_row_thresh)
            c_start = max(0, cX - col_thresh)
            c_end = min(ventricles.shape[1], cX + col_thresh)
            h_roi[r_start:mr_v, c_start:c_end] = 1

            ventricles_connected = np.logical_or(h_roi, ventricles)
            try:
                [r, c] = (1 - getLargestCC(measure.label(ventricles_connected, background=0))).nonzero()
            except (ValueError, IndexError):
                continue

            ventricles_connected_1 = ventricles.copy()
            ventricles_connected_1[r, c] = 0

            # Remove the lower part of the lateral ventricles
            fh_roi = np.ones(ventricles_connected_1.shape)
            fh_roi[y2:, :] = 0

            ventricles_fh = ventricles_connected_1 * fh_roi

            if np.sum(ventricles_fh) == 0:
                continue

            [vfh_r, vfh_c] = ventricles_fh.nonzero()
            if (max(vfh_r) - min(vfh_r)) > midline_detected_vert_thresh * (max(vfh_c) - min(vfh_c)):
                continue

            # Split left and right ventricles using ac_i +/- c_thresh
            # with iterative relaxation via dec_thresh
            slice_idx = max(0, ac_i_int - (c_thresh - dec_thresh))
            left_ven = ventricles_fh[:, :slice_idx]

            slice_idx_right = min(ventricles_fh.shape[1], ac_i_int + (c_thresh - dec_thresh))
            right_ven = ventricles_fh[:, slice_idx_right:]

            try:
                left_ven = getLargestCC(measure.label(left_ven, background=0))
            except (AssertionError, ValueError, IndexError):
                pass
            try:
                right_ven = getLargestCC(measure.label(right_ven, background=0))
            except (AssertionError, ValueError, IndexError):
                pass

            if visualize:
                plt.imshow(axial_gauss_blurred, cmap='gray')
                plt.title('Gauss blurred (raw MRI — no windowing)')
                plt.show()
                plt.imshow(ventricles_connected_1, cmap='gray')
                plt.title('Ventricles after connected component filtering')
                plt.show()
                plt.imshow(left_ven, cmap='gray')
                plt.title('Left ventricle')
                plt.show()
                plt.imshow(right_ven, cmap='gray')
                plt.title('Right ventricle')
                plt.show()

            [lr, lc] = left_ven.nonzero()
            [rr, rc] = right_ven.nonzero()

            if not (len(lr) > 0 and len(lc) > 0 and len(rr) > 0 and len(rc) > 0):
                continue

            # Measure max y-length of left frontal horn
            cur_max_l_fh_len = 0
            start_lc = min(lc)
            stop_lc = max(lc)
            for col in range(start_lc, stop_lc):
                try:
                    fh_len = y2 - min(lr[lc == col])
                except ValueError:
                    continue
                if fh_len > cur_max_l_fh_len:
                    cur_max_l_fh_len = fh_len
                    ly1 = min(lr[lc == col])
                    lx1 = int(np.round(np.mean(lc[lr == ly1])))

            # Measure max y-length of right frontal horn
            cur_max_r_fh_len = 0
            start_rc = min(rc)
            stop_rc = max(rc)
            for col in range(start_rc, stop_rc):
                try:
                    fh_len = y2 - min(rr[rc == col])
                except ValueError:
                    continue
                if fh_len > cur_max_r_fh_len:
                    cur_max_r_fh_len = fh_len
                    ry1 = min(rr[rc == col])
                    rx1 = int(np.round(np.mean(rc[rr == ry1])))

            if (cur_max_l_fh_len == 0) and (cur_max_r_fh_len == 0):
                continue

            fh_len_pl = 0  # resetting value for each plane
            if cur_max_l_fh_len >= cur_max_r_fh_len:
                fh_len_pl = y2 - ly1
                y1_pl = ly1
                x1_pl = lx1
            else:
                fh_len_pl = y2 - ry1
                y1_pl = ry1
                x1_pl = rx1 + ac_i_int + c_thresh - dec_thresh

            if fh_len_pl > max_fh_length:  # updating opt vals after each plane
                max_fh_length = fh_len_pl
                final_x1_opt, final_y1_opt = x1_pl, y1_pl
                full_brain_mask_opt = full_brain_mask
                opt_ven_image = ventricles_fh
                opt_left_ven = left_ven
                opt_right_ven = right_ven
                opt_plane = plane
                axial_pl_opt = axial_brain_im

        if max_fh_length == 0:
            # After all planes evaluated at this thresh and still 0, relax thresh
            dec_thresh = dec_thresh + 1
        else:
            # Compute normalization: max y-length of brain mask on optimal plane
            [fbr, fbc] = full_brain_mask_opt.nonzero()

            max_y_len = 0
            for col in sorted(np.unique(fbc)):
                y_len = max(fbr[fbc == col]) - min(fbr[fbc == col])
                if y_len > max_y_len:
                    max_y_len = y_len
                    fb_x = col
                    fb_y2 = max(fbr[fbc == col])
                    fb_y1 = min(fbr[fbc == col])

            ei_y = max_fh_length / max_y_len

            fig, ax = plt.subplots(figsize=(10, 10))
            ax.imshow(axial_pl_opt, cmap='gray')

            # Offset to make traced lengths clear if x coordinates are too close
            if abs(final_x1_opt - fb_x) < 4:
                final_x1_opt = final_x1_opt - 4

            [ax.scatter(x, y, marker='x', color='w', s=2)
             for x, y in zip(
                 [final_x1_opt] * (y2 - final_y1_opt),
                 [y for y in range(final_y1_opt, y2)]
             )]
            [ax.scatter(x, y, marker='x', color='w', s=2)
             for x, y in zip(
                 [fb_x] * (fb_y2 - fb_y1),
                 [y for y in range(fb_y1, fb_y2)]
             )]

            plt.title(f'EI-y = {ei_y:.4f}  (plane {opt_plane})')
            plt.show()

            return ei_y, opt_plane

## AC/PC coordinate loading

In [ ]:
def get_ac_for_subject(subject_id, img_rotated, dataset=DATASET):
    """
    Return the AC voxel coordinates as continuous indices in the rotated image.

    For OASIS: uses the hardcoded atlas AC voxel (OASIS_AC_VOXEL from mri_utils).
    For ABCD: reads a per-subject <subject_id>_acpc.txt file containing two lines
              of physical-space AC/PC coordinates (parsed by load_acpc_from_txt).

    Parameters
    ----------
    subject_id : str
        Subject identifier used to locate the AC/PC .txt file (ABCD only).
    img_rotated : sitk.Image
        The rotated (RAS-oriented) SimpleITK image. Used to convert physical
        point to continuous voxel index via TransformPhysicalPointToContinuousIndex.
    dataset : str
        Dataset name; controls which coordinate source is used.

    Returns
    -------
    ac_i : float
        Continuous column index of the AC.
    ac_j : float
        Continuous row index of the AC.
    ac_k : float
        Continuous slice index of the AC.
    """
    if dataset == "oasis1":
        # OASIS atlas is registered to the same template — use hardcoded voxel
        ac_phys = img_rotated.TransformIndexToPhysicalPoint(OASIS_AC_VOXEL)
        ac_i, ac_j, ac_k = img_rotated.TransformPhysicalPointToContinuousIndex(ac_phys)

    elif dataset == "abcd":
        # Per-subject ACPC txt: line 1 = AC physical coords, line 2 = PC physical coords
        txt_path = os.path.join(DATA_DIR, subject_id, f"{subject_id}_acpc.txt")
        if not os.path.exists(txt_path):
            raise FileNotFoundError(f"AC/PC txt not found: {txt_path}")
        ac_phys, pc_phys = load_acpc_from_txt(txt_path)
        ac_i, ac_j, ac_k = img_rotated.TransformPhysicalPointToContinuousIndex(ac_phys)

    else:
        raise NotImplementedError(
            f"AC loading not implemented for dataset={dataset!r}. "
            "Add your coordinate source here."
        )

    return ac_i, ac_j, ac_k

## Discover scans

In [ ]:
scan_files = sorted(glob.glob(os.path.join(DATA_DIR, SCAN_GLOB), recursive=True))

scan_df = pd.DataFrame({
    "subject_id": [SUBJECT_ID_FROM_PATH(p) for p in scan_files],
    "scan_path": scan_files,
})
print(f"Found {len(scan_df)} scans in {DATA_DIR}")
scan_df.head()

## Single-subject test

Run the full pipeline on one scan to verify orientation, AC coordinate loading, and EI-y extraction before batch processing.

**Key difference from CT:** After `rotate_image()` we pass `img_arr` directly to `calc_ei_y()`. There is no `window_stack_sitk()` call here — T1w intensities are used as-is.

In [ ]:
# Pick first subject
test_path = scan_df.iloc[0]['scan_path']
test_id = scan_df.iloc[0]['subject_id']
print(f"Testing on: {test_id}")
print(f"Path: {test_path}")

# Load and rotate (LPS -> RAS)
img = sitk.ReadImage(test_path)
img_rotated = rotate_image(img)

# Convert to array — raw MRI intensities, no windowing
img_arr = sitk.GetArrayFromImage(img_rotated)
print(f"Array shape: {img_arr.shape}")
print(f"Intensity range: [{img_arr.min()}, {img_arr.max()}]")
print(f"Mean intensity (non-zero): {img_arr[img_arr > 0].mean():.1f}")

# Show a mid-axial slice to verify orientation
mid_slice = img_arr.shape[0] // 2
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.set_title(f"Mid-axial slice (slice {mid_slice})")
ax1.imshow(img_arr[mid_slice, :, :], cmap='gray')
ax2.set_title("Intensity histogram (non-zero voxels)")
ax2.hist(img_arr[img_arr > 0].flatten(), bins=100)
ax2.set_xlabel("Intensity")
plt.tight_layout()
plt.show()

# Get AC continuous indices
ac_i, ac_j, ac_k = get_ac_for_subject(test_id, img_rotated)
print(f"AC continuous index — i (col): {ac_i:.2f}, j (row): {ac_j:.2f}, k (slice): {ac_k:.2f}")

# Translate so AC is at physical origin, recompute continuous index
ac_phys = img_rotated.TransformContinuousIndexToPhysicalPoint((ac_i, ac_j, ac_k))
img_translated = translate_image(img_rotated, 3, ac_phys)
axial_data = sitk.GetArrayFromImage(img_translated)
ac_i_t, ac_j_t, ac_k_t = img_translated.TransformPhysicalPointToContinuousIndex([0.0, 0.0, 0.0])
print(f"Translated AC continuous index — i: {ac_i_t:.2f}, j: {ac_j_t:.2f}, k: {ac_k_t:.2f}")

# Run EI-y with visualization
ei_y, opt_plane = calc_ei_y(axial_data, ac_i_t, ac_j_t, ac_k_t, ei_y_processing_params, visualize=True)
print(f"\nEI-y for {test_id}: {ei_y}")
print(f"Optimal plane: {opt_plane}")

## Batch processing

In [ ]:
results = []

for _, row in scan_df.iterrows():
    subject_id = row['subject_id']
    vol_path = row['scan_path']

    try:
        img = sitk.ReadImage(vol_path)
        img_rotated = rotate_image(img)

        # Get AC continuous index in rotated image
        ac_i, ac_j, ac_k = get_ac_for_subject(subject_id, img_rotated)

        # Translate so AC is at physical origin
        ac_phys = img_rotated.TransformContinuousIndexToPhysicalPoint((ac_i, ac_j, ac_k))
        img_translated = translate_image(img_rotated, 3, ac_phys)

        # Raw MRI array — no windowing (key MRI adaptation)
        axial_data = sitk.GetArrayFromImage(img_translated)

        # Re-derive continuous AC index after translation (should be near image center)
        ac_i_t, ac_j_t, ac_k_t = img_translated.TransformPhysicalPointToContinuousIndex([0.0, 0.0, 0.0])

        ei_y, opt_plane = calc_ei_y(
            axial_data, ac_i_t, ac_j_t, ac_k_t,
            ei_y_processing_params, visualize=False
        )

        results.append({
            "subject_id": subject_id,
            "ei_y": ei_y,
            "opt_plane": opt_plane,
        })
        print(f"{subject_id}: EI-y={ei_y}")

    except Exception as e:
        print(f"{subject_id}: ERROR — {e}")
        results.append({"subject_id": subject_id, "ei_y": None, "opt_plane": None})

results_df = pd.DataFrame(results)
print(f"\nProcessed {len(results_df)} subjects")
print(f"Successful: {results_df['ei_y'].notna().sum()}")
print(f"Failed: {results_df['ei_y'].isna().sum()}")

## Results analysis

In [ ]:
# Save CSV
output_path = os.path.join(OUTPUT_DIR, f"ei_y_{DATASET}.csv")
results_df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")

# Summary statistics
valid = results_df['ei_y'].dropna()
print(f"\nEI-y summary (n={len(valid)}):")
print(valid.describe().round(4))

# Distribution histogram
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(valid, bins=20, edgecolor='black')
ax.set_xlabel('EI-y')
ax.set_ylabel('Count')
ax.set_title(f'EI-y Distribution — {DATASET} (n={len(valid)})')
ax.axvline(x=0.4, color='r', linestyle='--', label='Clinical threshold (0.4)')
ax.legend()
plt.tight_layout()
plt.show()

# Flag extreme values for manual inspection
min_ei_y_lim = 0.1
max_ei_y_lim = 0.5
extreme = results_df[
    results_df['ei_y'].notna() &
    ((results_df['ei_y'] <= min_ei_y_lim) | (results_df['ei_y'] >= max_ei_y_lim))
]
print(f"\nSubjects with extreme EI-y (outside [{min_ei_y_lim}, {max_ei_y_lim}]): {len(extreme)}")
if len(extreme) > 0:
    print(extreme.to_string(index=False))